In [1]:
import sys; sys.path.append("../src")
from etl import geo_ingest as gi
import numpy as np, pandas as pd

g = gi.download("GSE28829", bronze="../data/bronze")
expr = g.pivot_samples("VALUE")
ph = g.phenotype_data
gpl = g.gpls["GPL570"].table
print(expr.shape, [c for c in gpl.columns if 'ymbol' in c])
print(expr.stack().describe().round(2).to_dict())

25-Nov-2026 10:33:22 DEBUG utils - Directory ../data/bronze already exists. Skipping.
25-Nov-2026 10:33:22 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE28nnn/GSE28829/soft/GSE28829_family.soft.gz to ../data/bronze\GSE28829_family.soft.gz
100%|██████████| 24.7M/24.7M [00:08<00:00, 3.10MB/s]
25-Nov-2026 10:33:31 DEBUG downloader - Size validation passed
25-Nov-2026 10:33:31 DEBUG downloader - Moving C:\Users\WINDOW~1\AppData\Local\Temp\tmpkha59nrn to C:\Users\Windows11\Documents\perio-athero-crosstalk\data\bronze\GSE28829_family.soft.gz
25-Nov-2026 10:33:31 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE28nnn/GSE28829/soft/GSE28829_family.soft.gz
25-Nov-2026 10:33:31 INFO GEOparse - Parsing ../data/bronze\GSE28829_family.soft.gz: 
25-Nov-2026 10:33:31 DEBUG GEOparse - DATABASE: GeoMiame
25-Nov-2026 10:33:31 DEBUG GEOparse - SERIES: GSE28829
25-Nov-2026 10:33:31 DEBUG GEOparse - PLATFORM: GPL570
C:\Users\Windows11\miniforge3\env

(54675, 29) ['Gene Symbol']
{'count': 1585575.0, 'mean': 5.27, 'std': 1.9, 'min': 2.5, '25%': 3.77, '50%': 4.74, '75%': 6.41, 'max': 14.14}


In [2]:
phen = ph['characteristics_ch1.1.phenotype']
pheno = pd.DataFrame({
    'sample_id': ph['geo_accession'],
    'group': np.where(phen.str.startswith('advanced'), 'disease', 'control'),
    'site': 'carotid artery', 'sex': np.nan, 'age': np.nan,
    'platform': ph['platform_id'],
}).reset_index(drop=True)
expr = expr[pheno['sample_id']]

mapping = gi.symbol_affy_u133(gpl)          # colonne "Gene Symbol"
genes, stats = gi.collapse_to_genes(expr, mapping)
print(stats)
print(pheno['group'].value_counts().to_dict())
print(genes.loc[genes.index.isin(['BTK', 'ITGAL'])].iloc[:, :3])
gi.validate(genes, pheno)
print(gi.write_silver("GSE28829", genes, pheno, stats, silver="../data/silver"))

{'probes_before': 54675, 'probes_annotated': 45782, 'genes': 22880}
{'disease': 16, 'control': 13}
       GSM714070  GSM714071  GSM714072
gene                                  
BTK     6.428627   6.906877   6.425667
ITGAL   5.347259   5.319476   5.420628
..\data\silver\GSE28829


In [3]:
import sys; sys.path.append("../src")
from etl import geo_ingest as gi
import numpy as np, pandas as pd

g = gi.download("GSE20129", bronze="../data/bronze")
ph = g.phenotype_data
ph = ph[ph['platform_id'] == 'GPL6104']                    # on garde seulement GPL6104
print(ph['characteristics_ch1.0.disease state'].value_counts().to_dict())

expr = g.pivot_samples("VALUE")[ph.index].dropna(how='any')
print("expr:", expr.shape)

pheno = pd.DataFrame({
    'sample_id': ph['geo_accession'],
    'group': np.where(ph['characteristics_ch1.0.disease state'] == 'atherosclerosis', 'disease', 'control'),
    'site': 'blood', 'sex': 'female', 'age': np.nan,
    'ethnicity': ph['characteristics_ch1.1.ethnicity'],
    'platform': ph['platform_id'],
}).reset_index(drop=True)
expr = expr[pheno['sample_id']]

gpl = g.gpls['GPL6104'].table
mapping = gi.symbol_affy_u133(gpl, col="Symbol")
genes, stats = gi.collapse_to_genes(expr, mapping)
print(stats)
print(genes.loc[genes.index.isin(['BTK', 'ITGAL'])].iloc[:, :3])
gi.validate(genes, pheno)
print(gi.write_silver("GSE20129", genes, pheno, stats, silver="../data/silver"))

25-Nov-2026 10:37:52 DEBUG utils - Directory ../data/bronze already exists. Skipping.
25-Nov-2026 10:37:52 INFO GEOparse - File already exist: using local version.
25-Nov-2026 10:37:52 INFO GEOparse - Parsing ../data/bronze\GSE20129_family.soft.gz: 
25-Nov-2026 10:37:52 DEBUG GEOparse - DATABASE: GeoMiame
25-Nov-2026 10:37:52 DEBUG GEOparse - SERIES: GSE20129
25-Nov-2026 10:37:52 DEBUG GEOparse - PLATFORM: GPL6104
25-Nov-2026 10:37:53 DEBUG GEOparse - PLATFORM: GPL10558
25-Nov-2026 10:37:56 DEBUG GEOparse - SAMPLE: GSM502400
25-Nov-2026 10:37:56 DEBUG GEOparse - SAMPLE: GSM502401
25-Nov-2026 10:37:56 DEBUG GEOparse - SAMPLE: GSM502402
25-Nov-2026 10:37:56 DEBUG GEOparse - SAMPLE: GSM502403
25-Nov-2026 10:37:56 DEBUG GEOparse - SAMPLE: GSM502404
25-Nov-2026 10:37:56 DEBUG GEOparse - SAMPLE: GSM502405
25-Nov-2026 10:37:57 DEBUG GEOparse - SAMPLE: GSM502406
25-Nov-2026 10:37:57 DEBUG GEOparse - SAMPLE: GSM502407
25-Nov-2026 10:37:57 DEBUG GEOparse - SAMPLE: GSM502408
25-Nov-2026 10:37:57 

{'no atherosclerosis': 71, 'atherosclerosis': 48}
expr: (22184, 119)
{'probes_before': 22184, 'probes_annotated': 22184, 'genes': 18197}
       GSM502400  GSM502401  GSM502402
gene                                  
BTK     6.722147   6.530728   6.570437
ITGAL   6.460193   7.053227   6.743467
..\data\silver\GSE20129
